
Complete Drift Detection Pipeline

Production-ready drift monitoring with PSI and KS tests



In [11]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Sample X_train data (reference data)
X_train = pd.DataFrame({
    'feature_a': np.random.normal(loc=0, scale=1, size=1000),
    'feature_b': np.random.normal(loc=5, scale=2, size=1000),
    'feature_c': np.random.randint(0, 10, size=1000)
})

# Sample X_production data (current data with some drift)
X_production = pd.DataFrame({
    'feature_a': np.random.normal(loc=0.5, scale=1.2, size=1000), # Drifted
    'feature_b': np.random.normal(loc=5, scale=2, size=1000), # Stable
    'feature_c': np.random.randint(2, 12, size=1000) # Drifted
})

In [12]:
import numpy as np
import pandas as pd
from scipy import stats
from datetime import datetime

# ============================================================
# DRIFT DETECTION: PSI + KS Test
# Use daily on production features to catch distribution shifts
# ============================================================

class DriftDetector:
  """Detect data drift using PSI and KS tests."""

  def __init__(self, reference_data, psi_threshold=0.1, ks_alpha=0.05):
    self.reference = reference_data
    self.psi_threshold = psi_threshold
    self.ks_alpha = ks_alpha
    self.history = []

  def calculate_psi(self, expected, actual, bins=10):
    """Population Stability Index for drift detection.
        PSI < 0.1: stable | 0.1-0.25: monitor | > 0.25: action
    """
    breakpoints = np.percentile(expected, np.linspace(0, 100, bins + 1))
    breakpoints[0], breakpoints[-1] = -np.inf, np.inf

    exp_pct = np.histogram(expected, breakpoints)[0] / len(expected)
    act_pct = np.histogram(actual, breakpoints)[0] / len(actual)

    # Avoid log(0): clip to small value
    exp_pct = np.clip(exp_pct, 0.0001, None)
    act_pct = np.clip(act_pct, 0.0001, None)

    return np.sum((act_pct - exp_pct) * np.log(act_pct / exp_pct))

  def check_drift(self, current_data):
    """Check all features for drift, return structured report"""
    report = {
        'timestamp': datetime.now().isoformat(),
        'features' : {},
        'alert_level' : 'OK' # OK, WARNING, CRITICAL
    }

    max_severity = 0
    for col in self.reference.columns:
      if col not in current_data.columns:
        continue

      ref_col = self.reference[col].dropna()
      cur_col = current_data[col].dropna()

      psi = self.calculate_psi(ref_col, cur_col)
      ks_stat, p_value = stats.ks_2samp(ref_col, cur_col)

      # detemine severity

      if psi > 0.25:
        severity = 'CRITICAL'
        max_severity = max(max_severity,2)
      elif psi > self.psi_threshold:
        severity = 'WARNING'
        max_severity = max(max_severity,1)
      else:
        severity = 'OK'

      report['features'][col] = {
          'psi' : round(psi, 4),
          'ks_statistic': round(ks_stat,4),
          'p_value' : round(p_value,4),
          'severity' : severity
      }

      report['alert_level'] = ['OK', 'WARNING', 'CRITICAL'][max_severity]
      self.history.append(report)
      return report

# Usage

detector = DriftDetector(X_train, psi_threshold=0.1)
report = detector.check_drift(X_production)
if report['alert_level'] != 'OK':
    for feat, m in report['features'].items():
        if m['severity'] != 'OK':
            print(f"[{m['severity']}] {feat}: PSI={m['psi']}")


[WARNING] feature_a: PSI=0.2348


Model Performance Monitor with Alerts

Track predictions, latency, accuracy, and trigger severity-based alerts




In [ ]:
import numpy as np
from collections import deque
from datetime import datetime
import time

# ============================================================
# MODEL MONITOR: Track predictions and trigger alerts
# Implements the three-pillar monitoring approach
# ============================================================

class ModelMonitor:
  """ Monitor Model performance in production..."""

  def __init__(self, window_size=1000):
    self.predictions = deque(maxlen=window_size)
    self.latencies = deque(maxlen=window_size)
    self.actuals = deque(maxlen=window_size)
    self.alerts = []

# Thresholds (tune based on your model)
    self.thresholds = {
        'latency_p99_ms': 100,
        'accuracy_min': 0.85
    }

def log_prediction(self, prediction, latency_ms, actual=None):
  """Log a single prediction with optional ground truth."""
  self.predictions.append(prediction)
  self.latencies.append(latency_ms)
  if actual is not None:
    self.actuals.append(prediction, actual)

def get_metrics(self):
  """ Calculate current monitoring metrics"""
  metrics = {
      'timestamp': datetime.now().isoformat(),
      'prediction_count' : len(self.predictions)
  }

  # latency metrics
  if self.latencies:
    lats = list(self.latencies)
    metrics['latency_p50'] = round(np.percentile(lats, 50), 1)
    metrics['latency_p99'] = round(np.percentile(lats, 99), 1)

  # accuracy (when labels available)

